# Experiment Two: Isotropic Singular Value Evolution
Analysis of the singular values and bias evolution with training for various isotropic-MLP layers. Enables consideration of an appropriate thresholding value for dynamic topology experiments



In [ ]:
from Dependencies import *
import torch.nn as nn
import torch
import numpy as np
import pickle as pkl
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

In [ ]:
# Hyperparameters
LEARNING_RATE = 5e-2 # Notably increased learning rate such that evolution is clearer (rerunning other experiments with 5e-2 did not show that isotropic-tanh underperforms particularly at this increased LR)
BATCH_SIZE = 48
DEVICE = try_gpu(output=True, i=0)
ARCHITECTURE = [3072, 100, 100, 10]
PSI_DECAY = 1e-1
ADAMW_WEIGHT_DECAY = 1e-3
TOTAL_EPOCHS = 40
REPEAT = 40
LAYER = 2
NORMALISATION = True
INTRINSIC_LENGTH_APPROACH = "TRAINABLE"
LINEAR_CORRECTION_APPROACH = "TRAINABLE+DECAY"
WEIGHT_INIT = "orthogonal"
SVD_BINS = 24
SAVE_DIR = f"./Saved_Models/Experiment 2/Singular Value Evolutions/{WEIGHT_INIT} (isotropic) 5e-2/"

os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# Found that elementwise normalisation of the dataset was generally beneficial as a preprocessing step
class PerPixelNormalize:
    def __init__(self):
        normaliser_dictionary = pkl.load(open("./CIFAR_normalisations.pkl", "rb"))
        self.mean = normaliser_dictionary["mean"].to(torch.float32)
        self.inv_std = normaliser_dictionary["inverse stddev"].to(torch.float32)

    def __call__(self, tensor):
        return (tensor - self.mean) * self.inv_std

In [ ]:
if NORMALISATION:
    print("Using Normalisation")
    transform = transforms.Compose([transforms.ToTensor(), PerPixelNormalize()])
else:
    print("Not using Normalisation")
    transform = transforms.Compose([transforms.ToTensor()])

# Get dataset
cifar_train = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
cifar_test = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

# DataLoaders
train_loader = DataLoader(cifar_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(cifar_test, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
def extract_svd_bias_snapshot(network):
    """
    Return per-layer [[singular_values, bias_vector], ...] using the current model
    parameters. Singular values are computed directly from each weight matrix.
    """
    snapshot = []
    with torch.no_grad():
        for weight, bias in zip(network.weight_parameters, network.bias_parameters):
            singular_values = torch.linalg.svdvals(weight.detach()).cpu().numpy().copy()
            bias_vector = bias.detach().cpu().numpy().copy()
            snapshot.append([singular_values, bias_vector])
    return snapshot


def load_ensemble_history_from_pickles(save_dir, repeats, total_epochs, layer):
    ensemble_singular_value_history = []
    ensemble_bias_history = []

    for i in range(1, repeats + 1):
        repeat_singular_value_history = []
        repeat_bias_history = []

        for epoch_idx in range(0, total_epochs + 1):
            file_path = os.path.join(save_dir, f"REPEAT_{i}_EPOCH_{epoch_idx}.pkl")
            with open(file_path, "rb") as f:
                snapshot = pkl.load(f)

            singular_values, bias_vector = snapshot[layer]
            repeat_singular_value_history.append(np.asarray(singular_values))
            repeat_bias_history.append(np.asarray(bias_vector))

        ensemble_singular_value_history.append(np.asarray(repeat_singular_value_history))
        ensemble_bias_history.append(np.asarray(repeat_bias_history))

    ensemble_singular_value_history = np.asarray(ensemble_singular_value_history)
    ensemble_bias_history = np.asarray(ensemble_bias_history)

    ensemble_singular_value_history = np.transpose(ensemble_singular_value_history, (1, 0, 2)).reshape(total_epochs + 1, -1)
    ensemble_bias_history = np.transpose(ensemble_bias_history, (1, 0, 2)).reshape(total_epochs + 1, -1)

    return ensemble_singular_value_history, ensemble_bias_history


In [ ]:
print({file.split("_E")[0].split("T_")[1] for file in os.listdir(SAVE_DIR)})

In [ ]:
ensemble_singular_value_history = []
ensemble_bias_history = []
FORCE_RERUN = False
for repeat in range(REPEAT):
    if (repeat+1) not in {int(file.split("_E")[0].split("T_")[1]) for file in os.listdir(SAVE_DIR)} or FORCE_RERUN:
        print(f"\n========== Repeat {repeat + 1}/{REPEAT} ==========")

        network = IsotropicTanhMLP(
            layers=ARCHITECTURE,
            flatten=True,
            unflatten_shape=None,
            intrinsic_length_approach=INTRINSIC_LENGTH_APPROACH,
            linear_correction_approach=LINEAR_CORRECTION_APPROACH,
            positive_intrinsic_length=True,
            init_intrinsic_length=1e-6,
            tanh_epsilon=1e-3,
            device=DEVICE,
            dtype=torch.get_default_dtype(),
        )

        network.simple_initialiser(weight_init=WEIGHT_INIT)
        network.to(DEVICE)

        optimiser = torch.optim.AdamW(network.parameters(), lr=LEARNING_RATE, weight_decay=ADAMW_WEIGHT_DECAY)
        loss = nn.CrossEntropyLoss()

        train_x, test_x = [], [0]
        train_cost, test_cost = [], []
        train_acc, test_acc = [], []

        network, temp_cost, temp_acc = testing_epoch(
            network, test_loader, DEVICE, loss, "classification"
        )
        test_cost.append(temp_cost)
        test_acc.append(temp_acc)

        initial_snapshot = extract_svd_bias_snapshot(network)
        with open(os.path.join(SAVE_DIR, f"REPEAT_{repeat + 1}_EPOCH_{0}.pkl"), "wb") as f:
            pkl.dump(initial_snapshot, f)


        singular_values, bias_vector = initial_snapshot[LAYER]
        singular_value_history = [np.asarray(singular_values)]
        bias_history = [np.asarray(bias_vector)]

        for epoch in range(TOTAL_EPOCHS):
            print(
                f"Repeat {repeat + 1}/{REPEAT} | Starting Epoch {epoch + 1}/{TOTAL_EPOCHS}... ",
                end=""
            )

            network, epoch_x, epoch_cost, epoch_acc = training_epoch(
                network=network,
                training_set=train_loader,
                device=DEVICE,
                optimiser=optimiser,
                loss=loss,
                current_epoch=epoch,
                classification_or_reconstruction="classification",
                lambda_psi=(
                    PSI_DECAY
                    if LINEAR_CORRECTION_APPROACH == "TRAINABLE+DECAY"
                    else 0.0
                ),
            )

            train_x += epoch_x
            train_cost += epoch_cost
            train_acc += epoch_acc

            network, temp_cost, temp_acc = testing_epoch(
                network, test_loader, DEVICE, loss, "classification"
            )
            test_x.append(epoch + 1)
            test_cost.append(temp_cost)
            test_acc.append(temp_acc)

            epoch_snapshot = extract_svd_bias_snapshot(network)

            with open(os.path.join(SAVE_DIR, f"REPEAT_{repeat + 1}_EPOCH_{epoch + 1}.pkl"), "wb") as f:
                pkl.dump(epoch_snapshot, f)

            singular_values, bias_vector = epoch_snapshot[LAYER]
            singular_value_history.append(np.asarray(singular_values))
            bias_history.append(np.asarray(bias_vector))

            print(f"Testing Cost: {test_cost[-1]:5.4f} Accuracy: {test_acc[-1]:5.4f}%")

        ensemble_singular_value_history.append(np.asarray(singular_value_history))
        ensemble_bias_history.append(np.asarray(bias_history))
    else: print(f"Skipping Repeat {repeat + 1}/{REPEAT}...")
ensemble_singular_value_history = np.asarray(ensemble_singular_value_history)
ensemble_bias_history = np.asarray(ensemble_bias_history)


Plotting of the results:

In [ ]:
import os
import pickle as pkl
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import cm

# ------------------------------------------------------------
# NeurIPS / paper-consistent styling
# Matches Experiment 3c typography and white-background settings.
# ------------------------------------------------------------
mpl.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "grid.linewidth": 0.5,
    "grid.alpha": 0.22,
    "lines.linewidth": 1.8,
    "savefig.dpi": 300,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------
os.makedirs("./TempImages", exist_ok=True)
out_svg = f"./TempImages/Experiment2_AllLayers_Ridgeplots_{REPEAT=}.svg"

# ------------------------------------------------------------
# Local helpers: loading
# ------------------------------------------------------------
def _load_ensemble_history_from_pickles(save_dir, repeats, total_epochs, layer):
    """
    Return pooled per-epoch singular-value and bias histories for a given layer.

    Output shapes:
        singular values: (total_epochs + 1, repeats * num_singular_values)
        biases:          (total_epochs + 1, repeats * num_bias_values)
    """
    ensemble_singular_value_history = []
    ensemble_bias_history = []

    for repeat_idx in range(1, repeats + 1):
        repeat_singular_value_history = []
        repeat_bias_history = []

        for epoch_idx in range(0, total_epochs + 1):
            file_path = os.path.join(
                save_dir,
                f"REPEAT_{repeat_idx}_EPOCH_{epoch_idx}.pkl"
            )

            with open(file_path, "rb") as f:
                snapshot = pkl.load(f)

            singular_values, bias_vector = snapshot[layer]
            repeat_singular_value_history.append(np.asarray(singular_values, dtype=np.float64))
            repeat_bias_history.append(np.asarray(bias_vector, dtype=np.float64))

        ensemble_singular_value_history.append(np.asarray(repeat_singular_value_history))
        ensemble_bias_history.append(np.asarray(repeat_bias_history))

    ensemble_singular_value_history = np.asarray(ensemble_singular_value_history, dtype=np.float64)
    ensemble_bias_history = np.asarray(ensemble_bias_history, dtype=np.float64)

    ensemble_singular_value_history = (
        np.transpose(ensemble_singular_value_history, (1, 0, 2))
        .reshape(total_epochs + 1, -1)
    )
    ensemble_bias_history = (
        np.transpose(ensemble_bias_history, (1, 0, 2))
        .reshape(total_epochs + 1, -1)
    )

    return ensemble_singular_value_history, ensemble_bias_history


# ------------------------------------------------------------
# Local helpers: transformed ridgeplot machinery
# ------------------------------------------------------------
def _gaussian_kernel1d(sigma: float, radius: int | None = None) -> np.ndarray:
    if sigma <= 0:
        return np.array([1.0], dtype=np.float64)

    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))

    x = np.arange(-radius, radius + 1, dtype=np.float64)
    k = np.exp(-(x ** 2) / (2 * sigma ** 2))
    k /= k.sum()
    return k


def _signed_log1p(x):
    x = np.asarray(x, dtype=np.float64)
    return np.sign(x) * np.log1p(np.abs(x))


def _log1p_forward(x):
    x = np.asarray(x, dtype=np.float64)
    return np.log1p(np.clip(x, 0.0, None))


def _default_epoch_step(n_epochs: int) -> int:
    if n_epochs <= 15:
        return 1
    elif n_epochs <= 40:
        return 2
    elif n_epochs <= 80:
        return 5
    return 10


def _prepare_epoch_distributions(value_history, transform):
    value_history = np.asarray(value_history, dtype=np.float64)

    if value_history.ndim != 2:
        raise ValueError("value_history must have shape (num_epochs, num_values)")

    transformed = []
    for epoch in range(value_history.shape[0]):
        vals = value_history[epoch]
        vals = vals[np.isfinite(vals)]

        if vals.size == 0:
            raise ValueError(f"Epoch {epoch} has no finite values.")

        transformed.append(transform(vals))

    return transformed


def _ensure_endpoints(raw_ticks, xmin, xmax):
    raw_ticks = np.asarray(raw_ticks, dtype=np.float64)
    raw_ticks = raw_ticks[(raw_ticks >= xmin) & (raw_ticks <= xmax)]
    raw_ticks = np.unique(np.concatenate(([xmin], raw_ticks, [xmax])))
    raw_ticks.sort()
    return raw_ticks


def _make_tick_positions_and_labels(mode, xmin, xmax, xticks=None):
    """
    xmin, xmax, and xticks are in raw units.
    Returned tick positions are in transformed coordinates.
    """
    if xticks is None:
        if mode == "singular":
            xticks = [0, 1, 2, 5, 10, 20, 50, 100, 200]
        elif mode == "bias":
            xticks = [-50, -20, -10, -5, -2, -1, 0, 1, 2, 5, 10, 20, 50]
        else:
            raise ValueError(mode)

    raw_ticks = _ensure_endpoints(xticks, xmin, xmax)

    if mode == "singular":
        pos = _log1p_forward(raw_ticks)
    elif mode == "bias":
        pos = _signed_log1p(raw_ticks)
    else:
        raise ValueError(mode)

    labels = [
        f"{int(t)}" if np.isclose(t, int(t)) else f"{t:g}"
        for t in raw_ticks
    ]

    return pos, labels


def _get_epoch_colors(cmap_name: str, n_epochs: int, lo: float = 0.18, hi: float = 0.98):
    cmap = cm.get_cmap(cmap_name)
    vals = np.linspace(lo, hi, n_epochs)
    return [cmap(v) for v in vals]


def _plot_transformed_ridgeline_panel(
    ax,
    value_history,
    *,
    mode,
    title,
    xlabel,
    xmin,
    xmax,
    xticks=None,
    bins=100,
    smooth_sigma_bins=1.0,
    ridge_height=2.45,
    row_step=0.38,
    cmap_name="YlGnBu",
    epoch_step=5,
    baseline_color="#8f8f8f",
    baseline_alpha=0.28,
    outline_color="white",
    outline_alpha=0.72,
    outline_width=0.95,
    fill_alpha=0.94,
    zero_line=True,
):
    """
    Ridgeline panel using the same transformed-coordinate logic as the
    existing Experiment 2 figure.

    xmin, xmax, and xticks are specified in raw units.
    Histograms are computed in transformed x-space.
    """
    if mode == "singular":
        transform = _log1p_forward
        if xmin < 0:
            raise ValueError("For singular values, xmin must be >= 0.")
    elif mode == "bias":
        transform = _signed_log1p
    else:
        raise ValueError("mode must be 'singular' or 'bias'")

    transformed_epochs = _prepare_epoch_distributions(value_history, transform)
    n_epochs = len(transformed_epochs)

    zmin = float(transform(np.array([xmin]))[0])
    zmax = float(transform(np.array([xmax]))[0])

    edges = np.linspace(zmin, zmax, bins + 1)
    centres = 0.5 * (edges[:-1] + edges[1:])

    density_matrix = np.zeros((n_epochs, bins), dtype=np.float64)

    for e in range(n_epochs):
        vals = transformed_epochs[e]
        vals_in = vals[(vals >= zmin) & (vals <= zmax)]

        if vals_in.size == 0:
            density = np.zeros(bins, dtype=np.float64)
        else:
            density, _ = np.histogram(vals_in, bins=edges, density=True)

            # Preserve visible-window mass scaling from the existing plot.
            mass_frac = vals_in.size / vals.size
            density = density * mass_frac

        density_matrix[e] = density

    if smooth_sigma_bins > 0:
        kernel = _gaussian_kernel1d(sigma=smooth_sigma_bins)
        for e in range(n_epochs):
            density_matrix[e] = np.convolve(density_matrix[e], kernel, mode="same")

    max_density = float(np.max(density_matrix))
    if max_density <= 0:
        max_density = 1.0

    y_positions = np.arange(n_epochs)[::-1]
    colors = _get_epoch_colors(cmap_name, n_epochs, lo=0.18, hi=0.98)

    for e in range(n_epochs):
        baseline = y_positions[e] * row_step
        dens = density_matrix[e]
        y = baseline + ridge_height * dens / max_density
        color = colors[e]

        ax.hlines(
            baseline,
            xmin=zmin,
            xmax=zmax,
            color=baseline_color,
            linewidth=0.7,
            alpha=baseline_alpha,
            zorder=1,
        )

        ax.fill_between(
            centres,
            baseline,
            y,
            color=color,
            alpha=fill_alpha,
            linewidth=0,
            zorder=2 + e,
        )

        ax.plot(
            centres,
            y,
            color=outline_color,
            linewidth=outline_width,
            alpha=outline_alpha,
            zorder=3 + e,
        )

    # Keep the same reference-line placement as the existing Experiment 2 plot,
    # but make it printable on a white background.
    if mode == "bias" and zero_line:
        ax.axvline(
            0.0,
            linestyle=(0, (4, 3)),
            color="0.35",
            linewidth=0.9,
            alpha=0.65,
            zorder=0,
        )
    elif mode == "singular" and zero_line:
        ax.axvline(
            np.log(2),
            linestyle=(0, (4, 3)),
            color="0.35",
            linewidth=0.9,
            alpha=0.65,
            zorder=0,
        )

    if epoch_step is None:
        epoch_step = _default_epoch_step(n_epochs)

    shown_epochs = np.arange(0, n_epochs, epoch_step)
    if shown_epochs[-1] != n_epochs - 1:
        shown_epochs = np.append(shown_epochs, n_epochs - 1)

    tick_positions = y_positions[shown_epochs] * row_step
    ax.set_yticks(tick_positions)
    ax.set_yticklabels(shown_epochs)

    xtick_positions, xtick_labels = _make_tick_positions_and_labels(
        mode=mode,
        xmin=xmin,
        xmax=xmax,
        xticks=xticks,
    )
    ax.set_xticks(xtick_positions)
    ax.set_xticklabels(xtick_labels)

    ax.set_xlim(zmin, zmax)
    ax.set_ylim(-0.12, y_positions[0] * row_step + ridge_height + 0.22)

    ax.set_title(title, pad=10)
    ax.set_xlabel(xlabel)

    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0, width=0.8, colors="black")
    ax.tick_params(axis="x", direction="out", length=3, width=0.8, colors="black")
    ax.grid(False)
    ax.margins(x=0)
    ax.set_facecolor("white")


# ------------------------------------------------------------
# Main plot: all layers in one row
# ------------------------------------------------------------
layer_cmaps = {
    0: "YlGnBu",
    1: "YlOrBr",
    2: "PuBuGn",
}

panel_specs = [
    (0, "singular", "(a) Layer 0 - Weight Spectra"),
    (0, "bias",     "(b) Layer 0 - Bias Spectra"),
    (1, "singular", "(c) Layer 1 - Weight Spectra"),
    (1, "bias",     "(d) Layer 1 - Bias Spectra"),
    (2, "singular", "(e) Layer 2 - Weight Spectra"),
    (2, "bias",     "(f) Layer 2 - Bias Spectra"),
]

fig, axes = plt.subplots(
    1, 6,
    figsize=(22.0, 4.8),
    sharey=True,
    constrained_layout=True,
    gridspec_kw={"width_ratios": [1, 1, 1, 1, 1, 1]},
)
fig.patch.set_facecolor("white")

# Load each layer once, then reuse for its two panels.
layer_histories = {}
for layer in layer_cmaps:
    layer_histories[layer] = _load_ensemble_history_from_pickles(
        save_dir=SAVE_DIR,
        repeats=REPEAT,
        total_epochs=TOTAL_EPOCHS,
        layer=layer,
    )

for ax_idx, (layer, mode, title) in enumerate(panel_specs):
    ax = axes[ax_idx]
    singular_history, bias_history = layer_histories[layer]

    if mode == "singular":
        _plot_transformed_ridgeline_panel(
            ax,
            singular_history,
            mode="singular",
            title=title,
            xlabel="Singular value",
            xmin=0,
            xmax=[1500, None, 200, None, 100, None][ax_idx],
            xticks=[0, 1, 2, 5, 10, 20, 50, 100, 250, 500, 1500],
            bins=100,
            smooth_sigma_bins=1.0,
            ridge_height=2.45,
            row_step=0.38,
            epoch_step=5,
            cmap_name=layer_cmaps[layer],
            baseline_color="#8aa7c7",
            baseline_alpha=0.22,
            outline_color="white",
            outline_alpha=0.72,
            outline_width=0.95,
            fill_alpha=0.94,
            zero_line=True,
        )
    else:
        _plot_transformed_ridgeline_panel(
            ax,
            bias_history,
            mode="bias",
            title=title,
            xlabel="Bias",
            xmin=[None, -250, None, -5, None, -15][ax_idx],
            xmax=[None, 250, None, 5, None, 15][ax_idx],
            xticks=[-250, -100, -50, -20, -10, -5, -2, -1, 0, 1, 2, 5, 10, 20, 50, 100, 250],
            bins=100,
            smooth_sigma_bins=1.0,
            ridge_height=2.45,
            row_step=0.38,
            epoch_step=5,
            cmap_name=layer_cmaps[layer],
            baseline_color="#d3a47e",
            baseline_alpha=0.24,
            outline_color="white",
            outline_alpha=0.72,
            outline_width=0.95,
            fill_alpha=0.94,
            zero_line=True,
        )

    if ax_idx == 0:
        ax.set_ylabel("Epoch")
    else:
        ax.set_ylabel("")
        ax.tick_params(axis="y", labelleft=False)

# No footer text: intentionally removed for the NeurIPS version.
fig.savefig(out_svg, bbox_inches="tight", facecolor="white")
plt.show()

print({
    "out_svg": out_svg,
    "n_repeats": REPEAT,
    "total_epochs": TOTAL_EPOCHS,
    "layers": list(layer_cmaps.keys()),
})